# 21 · `portiere replay --auto-replay` Walkthrough (v0.3.1)

Walk through Portiere's v0.3.1 auto-replay feature:

1. Build a synthetic `manifest.lock.json` with recorded stages.
2. Call `auto_replay()` to re-run each stage and compare outputs against the manifest.
3. Inspect the per-stage `ReplayReport`.

Tolerance bands (locked in the v0.3.1 plan):

- **ingest** — format-equality (source hash verified upstream)
- **schema** — `n_mappings` identical
- **concept** — ±1% drift on `auto_rate`; `n_mappings` must not drop
- **etl** — `row_count` + column set identical (paths can differ)
- **validate** — `all_passed` binary equality

Stages whose dependencies aren't available in the current environment record as `UNAVAILABLE` and do not fail the report. Full BYO-LLM rehydration is a v0.3.x roadmap item.

Reference: [`docs/reproducibility.md`](../reproducibility.md).

In [ ]:
import json, hashlib, shutil
from pathlib import Path

import portiere
from portiere.repro.replay import auto_replay

print('portiere version:', portiere.__version__)

## 1 · Build a synthetic project + manifest

We materialize a source CSV, compute its sha256, and write a manifest claiming two pipeline stages were recorded: `ingest` and `validate`. The other stages (schema / concept / etl) will be marked `UNAVAILABLE` because their LLM-bound re-execution is out of scope for v0.3.1.

In [ ]:
work_dir = Path('/tmp/portiere-replay-nb21')
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir()

# Source CSV the manifest will reference
src = work_dir / 'patients.csv'
src.write_text('patient_id,dob\n1,1990-01-15\n2,1985-07-20\n')
src_sha = hashlib.sha256(src.read_bytes()).hexdigest()
print(f'Source: {src}\n  sha256: {src_sha[:16]}...')

In [ ]:
manifest = {
    'manifest_version': '1',
    'run': {
        'run_id': 'demo-21',
        'started_at': '2026-05-12T00:00:00+00:00',
        'finished_at': '2026-05-12T00:01:00+00:00',
        'duration_seconds': 60.0,
    },
    'portiere_version': portiere.__version__,
    'python_version': '3.12.1',
    'os_string': 'DemoOS',
    'git_sha': None,
    'git_dirty': None,
    'project_name': 'nb21-demo',
    'target_model': 'omop_cdm_v5.4',
    'task': 'standardize',
    'source_standard': None,
    'vocabularies_requested': [],
    'embedding': {'name': 'sapbert', 'dimension': 768, 'hf_revision': None, 'sha256_of_config': None},
    'knowledge_backend': None,
    'vocabularies': [],
    'prompt_templates': [],
    'thresholds': {},
    'source_data': {
        'path': str(src),
        'sha256': src_sha,
        'connection_string_redacted': None,
        'table_or_query': None,
    },
    'stages': [
        {
            'stage': 'ingest',
            'started_at': '2026-05-12T00:00:01+00:00',
            'finished_at': '2026-05-12T00:00:02+00:00',
            'inputs': {'source_name': 'patients', 'format': 'csv'},
            'outputs': {'row_count': 2, 'schema': ['patient_id', 'dob']},
            'metrics': {},
        },
        {
            'stage': 'schema',
            'started_at': '2026-05-12T00:00:03+00:00',
            'finished_at': '2026-05-12T00:00:05+00:00',
            'inputs': {'source_name': 'patients'},
            'outputs': {'n_mappings': 2},
            'metrics': {},
        },
    ],
}

manifest_path = work_dir / 'manifest.lock.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f'Manifest: {manifest_path}')

## 2 · Run `auto_replay()`

Verifies artifacts, reconstructs the project, then walks `manifest.stages` and compares each replayed output against what the manifest recorded.

In [ ]:
report = auto_replay(manifest_path)

print(f'Manifest:     {report.manifest_path}')
print(f'Overall pass: {report.passed}')
print(f'Stages:       {len(report.per_stage)}')
print()
for s in report.per_stage:
    status = (
        'PASS' if s.passed is True
        else 'FAIL' if s.passed is False
        else 'UNAVAILABLE'
    )
    drift = f' drift={s.drift_pct:.2f}%' if s.drift_pct is not None else ''
    reason = f' ({s.reason})' if s.reason else ''
    print(f'  [{status:<11}] {s.stage}{drift}{reason}')

## 3 · What just happened

- **ingest** PASSED — the comparator pulled the file's current format (`.csv`) and matched it against the manifest's recorded `format: csv`.
- **schema** is `UNAVAILABLE` — re-running schema mapping requires the same LLM the original run used. v0.3.1's auto-replay records that honestly instead of producing a stale comparison.

If we had added a `validate` stage with `metrics.all_passed = true` and the recorded `output_path` still existed, the comparator would re-run `project.validate()` and assert the `all_passed` flag matches.

## 4 · CLI form

Same thing from the shell:

```bash
portiere replay --auto-replay /tmp/portiere-replay-nb21/manifest.lock.json
```

Exit codes:

- `0` — all stages passed or were `UNAVAILABLE`
- `1` — at least one stage failed comparison
- `2` — artifact-verification failed (source/vocab missing or sha mismatch)

In [ ]:
# Tidy up
shutil.rmtree(work_dir)
print('Cleaned up.')